In [0]:
# Configuración de credenciales para acceso al Blob Storage

spark.conf.set(
    "fs.azure.account.key.diplomaturastorage2026.blob.core.windows.net",
    "TU_KEY_AQUI"
)

In [0]:
# Definición de rutas Silver y Gold

ruta_silver = "wasbs://silver@diplomaturastorage2026.blob.core.windows.net/"
ruta_gold = "wasbs://gold@diplomaturastorage2026.blob.core.windows.net/"

In [0]:
# Lectura de tabla Silver

df_ventas_silver = spark.read \
    .format("delta") \
    .load(ruta_silver + "ventas")

In [0]:
# Validación inicial de Silver

print("Cantidad registros Silver:", df_ventas_silver.count())

df_ventas_silver.printSchema()

Cantidad registros Silver: 2000
root
 |-- id_tienda: integer (nullable = true)
 |-- id_cliente: integer (nullable = true)
 |-- id_producto: integer (nullable = true)
 |-- id_venta: integer (nullable = true)
 |-- fecha_venta: date (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio_unitario: decimal(10,2) (nullable = true)
 |-- nombre_producto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- marca: string (nullable = true)
 |-- nombre_cliente: string (nullable = true)
 |-- genero: string (nullable = true)
 |-- edad: integer (nullable = true)
 |-- nombre_tienda: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- provincia: string (nullable = true)
 |-- importe_total: decimal(21,2) (nullable = true)
 |-- segmento_edad: string (nullable = true)
 |-- tipo_ticket: string (nullable = true)
 |-- cantidad_compras_cliente: long (nullable = true)
 |-- tipo_cliente: string (nullable = true)



In [0]:
# Creación de vista temporal SQL

df_ventas_silver.createOrReplaceTempView("ventas_silver")

In [0]:
%sql
-- Análisis de ventas por provincia

SELECT
    provincia,
    SUM(importe_total) AS ventas_totales,
    COUNT(DISTINCT id_venta) AS cantidad_ventas,
    COUNT(DISTINCT id_cliente) AS clientes_unicos
FROM ventas_silver
GROUP BY provincia
ORDER BY ventas_totales DESC

provincia,ventas_totales,cantidad_ventas,clientes_unicos
Buenos Aires,22333682.00,775,100
Mendoza,12061025.00,439,100
Córdoba,11985606.00,389,99
Santa Fe,11503619.00,397,99


In [0]:
%sql
-- Análisis de ventas por producto

SELECT
    nombre_producto,
    marca,
    SUM(importe_total) AS ventas_totales,
    COUNT(DISTINCT id_venta) AS cantidad_ventas,
    COUNT(DISTINCT id_cliente) AS clientes_unicos
FROM ventas_silver
GROUP BY nombre_producto, marca
ORDER BY ventas_totales DESC

nombre_producto,marca,ventas_totales,cantidad_ventas,clientes_unicos
Auriculares,Sony,6797907.00,221,91
Celular,Motorola,6297475.00,211,90
Parlante,JBL,6042583.00,199,88
Mouse,Logitech,5967359.00,207,86
Tablet,Apple,5870408.00,214,89
Impresora,HP,5585664.00,196,83
Teclado,Redragon,5522230.00,189,86
Webcam,Logitech,5515208.00,200,90
Monitor,Samsung,5168739.00,188,81
Notebook,Lenovo,5116359.00,175,84


In [0]:
%sql
-- Análisis de ventas por sucursal

SELECT
    nombre_tienda,
    provincia,
    SUM(importe_total) AS ventas_totales,
    COUNT(DISTINCT id_venta) AS cantidad_ventas,
    COUNT(DISTINCT id_cliente) AS clientes_unicos
FROM ventas_silver
GROUP BY nombre_tienda, provincia
ORDER BY ventas_totales DESC

nombre_tienda,provincia,ventas_totales,cantidad_ventas,clientes_unicos
Sucursal Oeste,Mendoza,12061025.00,439,100
Sucursal Este,Buenos Aires,12054062.00,400,100
Sucursal Norte,Córdoba,11985606.00,389,99
Sucursal Sur,Santa Fe,11503619.00,397,99
Sucursal Centro,Buenos Aires,10279620.00,375,97


In [0]:
%sql
-- Evolución temporal de ventas por fecha

SELECT
    fecha_venta,
    SUM(importe_total) AS ventas_totales,
    COUNT(DISTINCT id_venta) AS cantidad_ventas
FROM ventas_silver
GROUP BY fecha_venta
ORDER BY fecha_venta

fecha_venta,ventas_totales,cantidad_ventas
2026-01-01,375988.00,16
2026-01-02,386687.00,14
2026-01-03,449652.00,15
2026-01-04,294439.00,11
2026-01-05,283435.00,13
2026-01-06,392967.00,9
2026-01-07,342110.00,15
2026-01-08,313758.00,14
2026-01-09,473798.00,16
2026-01-10,319948.00,12


In [0]:
%sql
-- Análisis de ventas por género

SELECT
    genero,
    SUM(importe_total) AS ventas_totales,
    COUNT(DISTINCT id_cliente) AS clientes_unicos,
    COUNT(DISTINCT id_venta) AS cantidad_ventas
FROM ventas_silver
GROUP BY genero
ORDER BY ventas_totales DESC

genero,ventas_totales,clientes_unicos,cantidad_ventas
Masculino,35806459.00,62,1228
Femenino,22077473.00,38,772


In [0]:
%sql
-- Análisis de ventas por género

SELECT
    edad,
    SUM(importe_total) AS ventas_totales,
    COUNT(DISTINCT id_cliente) AS clientes_unicos,
    COUNT(DISTINCT id_venta) AS cantidad_ventas
FROM ventas_silver
GROUP BY edad
ORDER BY ventas_totales DESC

edad,ventas_totales,clientes_unicos,cantidad_ventas
20,3470686.00,5,103
62,3059932.00,4,94
65,2902415.00,4,94
56,2617800.00,5,101
69,2197961.00,3,64
51,2146136.00,3,71
53,1859047.00,3,58
49,1802413.00,3,69
19,1755945.00,3,58
58,1742652.00,3,62


In [0]:
# Construcción de tabla Gold por provincia

df_gold_provincia = spark.sql("""
SELECT
    provincia,
    SUM(importe_total) AS ventas_totales,
    COUNT(DISTINCT id_venta) AS cantidad_ventas,
    COUNT(DISTINCT id_cliente) AS clientes_unicos
FROM ventas_silver
GROUP BY provincia
""")

display(df_gold_provincia)

provincia,ventas_totales,cantidad_ventas,clientes_unicos
Mendoza,12061025.00,439,100
Santa Fe,11503619.00,397,99
Buenos Aires,22333682.00,775,100
Córdoba,11985606.00,389,99


In [0]:
# Escritura de tabla Gold por provincia

df_gold_provincia.write \
    .format("delta") \
    .mode("overwrite") \
    .save(ruta_gold + "ventas_provincia")

In [0]:
# Construcción de tabla Gold por producto

df_gold_producto = spark.sql("""
SELECT
    nombre_producto,
    marca,
    SUM(importe_total) AS ventas_totales,
    COUNT(DISTINCT id_venta) AS cantidad_ventas,
    COUNT(DISTINCT id_cliente) AS clientes_unicos
FROM ventas_silver
GROUP BY nombre_producto, marca
""")

display(df_gold_producto)

nombre_producto,marca,ventas_totales,cantidad_ventas,clientes_unicos
Impresora,HP,5585664.00,196,83
Webcam,Logitech,5515208.00,200,90
Monitor,Samsung,5168739.00,188,81
Auriculares,Sony,6797907.00,221,91
Teclado,Redragon,5522230.00,189,86
Celular,Motorola,6297475.00,211,90
Notebook,Lenovo,5116359.00,175,84
Parlante,JBL,6042583.00,199,88
Mouse,Logitech,5967359.00,207,86
Tablet,Apple,5870408.00,214,89


In [0]:
# Escritura de tabla Gold por producto

df_gold_producto.write \
    .format("delta") \
    .mode("overwrite") \
    .save(ruta_gold + "ventas_producto")

In [0]:
# Construcción de tabla Gold por tienda

df_gold_tienda = spark.sql("""
SELECT
    nombre_tienda,
    provincia,
    SUM(importe_total) AS ventas_totales,
    COUNT(DISTINCT id_venta) AS cantidad_ventas,
    COUNT(DISTINCT id_cliente) AS clientes_unicos
FROM ventas_silver
GROUP BY nombre_tienda, provincia
""")

display(df_gold_tienda)

nombre_tienda,provincia,ventas_totales,cantidad_ventas,clientes_unicos
Sucursal Oeste,Mendoza,12061025.00,439,100
Sucursal Este,Buenos Aires,12054062.00,400,100
Sucursal Centro,Buenos Aires,10279620.00,375,97
Sucursal Sur,Santa Fe,11503619.00,397,99
Sucursal Norte,Córdoba,11985606.00,389,99


In [0]:
# Escritura de tabla Gold por tienda

df_gold_tienda.write \
    .format("delta") \
    .mode("overwrite") \
    .save(ruta_gold + "ventas_tienda")

In [0]:
# Construcción de tabla Gold por segmento etario

df_gold_segmento_edad = spark.sql("""
SELECT
    segmento_edad,
    SUM(importe_total) AS ventas_totales,
    COUNT(DISTINCT id_venta) AS cantidad_ventas,
    COUNT(DISTINCT id_cliente) AS clientes_unicos
FROM ventas_silver
GROUP BY segmento_edad
""")

display(df_gold_segmento_edad)

segmento_edad,ventas_totales,cantidad_ventas,clientes_unicos
Senior,28321484.00,989,49
Joven,11240877.00,362,19
Adulto,18321571.00,649,32


In [0]:
# Escritura de tabla Gold por segmento etario

df_gold_segmento_edad.write \
    .format("delta") \
    .mode("overwrite") \
    .save(ruta_gold + "ventas_segmento_edad")

In [0]:
# Verificación de tablas almacenadas en Gold

display(dbutils.fs.ls(ruta_gold))

path,name,size,modificationTime
wasbs://gold@diplomaturastorage2026.blob.core.windows.net/ventas_producto/,ventas_producto/,0,1778593750000
wasbs://gold@diplomaturastorage2026.blob.core.windows.net/ventas_provincia/,ventas_provincia/,0,1778593746000
wasbs://gold@diplomaturastorage2026.blob.core.windows.net/ventas_segmento_edad/,ventas_segmento_edad/,0,1778593762000
wasbs://gold@diplomaturastorage2026.blob.core.windows.net/ventas_tienda/,ventas_tienda/,0,1778593755000


In [0]:
# Visualización gerencial: ventas totales por provincia

df_visual_provincia = spark.read \
    .format("delta") \
    .load(ruta_gold + "ventas_provincia")

display(df_visual_provincia)

provincia,ventas_totales,cantidad_ventas,clientes_unicos
Mendoza,12061025.00,439,100
Santa Fe,11503619.00,397,99
Buenos Aires,22333682.00,775,100
Córdoba,11985606.00,389,99


Databricks visualization. Run in Databricks to view.

In [0]:
# Visualización gerencial: ventas totales por producto

df_visual_producto = spark.read \
    .format("delta") \
    .load(ruta_gold + "ventas_producto")

display(df_visual_producto)

nombre_producto,marca,ventas_totales,cantidad_ventas,clientes_unicos
Impresora,HP,5585664.00,196,83
Webcam,Logitech,5515208.00,200,90
Monitor,Samsung,5168739.00,188,81
Auriculares,Sony,6797907.00,221,91
Teclado,Redragon,5522230.00,189,86
Celular,Motorola,6297475.00,211,90
Notebook,Lenovo,5116359.00,175,84
Parlante,JBL,6042583.00,199,88
Mouse,Logitech,5967359.00,207,86
Tablet,Apple,5870408.00,214,89


Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import sum

# Visualización gerencial: evolución temporal de ventas

df_visual_fecha = df_ventas_silver.groupBy("fecha_venta").agg(
    sum("importe_total").alias("ventas_totales")
).orderBy("fecha_venta")

display(df_visual_fecha)

fecha_venta,ventas_totales
2026-01-01,375988.00
2026-01-02,386687.00
2026-01-03,449652.00
2026-01-04,294439.00
2026-01-05,283435.00
2026-01-06,392967.00
2026-01-07,342110.00
2026-01-08,313758.00
2026-01-09,473798.00
2026-01-10,319948.00


Databricks visualization. Run in Databricks to view.

In [0]:
# Visualización gerencial: ventas por segmento etario

df_visual_segmento = spark.read \
    .format("delta") \
    .load(ruta_gold + "ventas_segmento_edad")

display(df_visual_segmento)

segmento_edad,ventas_totales,cantidad_ventas,clientes_unicos
Senior,28321484.00,989,49
Joven,11240877.00,362,19
Adulto,18321571.00,649,32


Databricks visualization. Run in Databricks to view.

In [0]:
df_gold_segmento_edad.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ventas_segmento_edad")